In [0]:
from pyspark.sql import functions as F

def normalize_columns(df):
    """Standardizes column names: lowercase, spaces to underscores."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

In [0]:
df_invoice = spark.table("workspace.bronze.invoice")
df_invoice = normalize_columns(df_invoice)

df_invoice = (
    df_invoice
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("invoice_amount", F.col("invoice_amount").cast("double"))
    .dropDuplicates()
)

df_invoice.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.invoice")
print(f"silver.invoice: {df_invoice.count()} rows written")